# Getting Started Notebook

This notebook mirrors the shortest useful `ffTRF` workflow: create a model, fit it, inspect it, and score predictions. It is intentionally close to the package examples so users can move between the docs site and runnable scripts without translating the API.

In [ ]:
import numpy as np

from fftrf import TRF, inverse_variance_weights

rng = np.random.default_rng(0)
fs = 1_000

stimulus = [rng.standard_normal((8_000, 3)) for _ in range(4)]
response = [rng.standard_normal((8_000, 2)) for _ in range(4)]

model = TRF(direction=1)
cv_scores = model.train(
    stimulus=stimulus,
    response=response,
    fs=fs,
    tmin=-0.050,
    tmax=0.250,
    regularization=np.logspace(-6, 1, 8),
    segment_duration=2.048,
    overlap=0.5,
    window="hann",
    k="loo",
    trial_weights=inverse_variance_weights(response),
)


## What to inspect after fitting

- `model.weights`: lag-domain kernel over the requested `tmin` to `tmax` window
- `model.transfer_function`: complex frequency-domain solution
- `model.times`: lag axis in seconds
- `model.frequencies`: frequency axis in Hz
- `model.regularization`: selected regularization value after cross-validation
- `cv_scores`: cross-validation scores for each candidate value

If you already know the regularization you want, pass a single scalar and `train(...)` will fit directly instead of running a candidate search.

## Notebook-generated plots

The figures below are generated from the model fitted in this notebook, so they stay aligned with the exact code and parameters shown here.


In [ ]:
import matplotlib.pyplot as plt

prediction, score = model.predict(stimulus=stimulus, response=response, average=False)
score = np.asarray(score, dtype=float)
times = np.arange(response[0].shape[0], dtype=float) / fs

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
model.plot(
    input_index=0,
    output_index=0,
    ax=axes[0],
    title="Recovered kernel: input 1 -> output 1",
)

axes[1].plot(times, response[0][:, 0], color="#222222", linewidth=1.4, label="Observed")
axes[1].plot(
    times,
    np.asarray(prediction[0], dtype=float)[:, 0],
    color="#0B6E4F",
    linewidth=1.2,
    label="Predicted",
)
axes[1].set_title(f"Prediction trace on trial 1 (output 1, r={score[0]:.3f})")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Amplitude (z)")
axes[1].grid(alpha=0.2, linewidth=0.6)
_ = axes[1].legend(frameon=False)
plt.show()


## Related next steps

- Read the full [Getting Started](../../getting-started/) page for parameter-by-parameter guidance.
- Walk through the intended lifecycle in [Core Workflow](../../guides/core-workflow/).
- Open [Examples](../../examples/) to find the script that most closely matches your data.
- Use `plot_grid(...)`, `score(...)`, `cross_spectral_diagnostics(...)`, or `bootstrap_confidence_interval(...)` once the first fit is working.
